# StickVoice → AI video clips (free, open-source Wan 2.1)

Turns the `clips.json` that StickVoice writes next to every video into real AI video clips, using the
open-source **Wan 2.1 text-to-video 1.3B** model (Apache-2.0) on Colab's free GPU.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. When asked, upload `clips.json` from your `output\<video>\` folder.
3. Wait (about 5–15 min per clip on a free T4). At the end `clips.zip` downloads.
4. On your PC: put `clips.zip` in the same `output\<video>\` folder and run
   `assemble_clips.bat` (or `python -m stickvoice --assemble output\<video>`). You get `ai_video.mp4`
   with your StickVoice voice and captions.

Turn on **SAVE_TO_DRIVE** to keep finished clips in Google Drive: if Colab disconnects, run all again and it continues
where it stopped.

In [ ]:
# Settings
SAVE_TO_DRIVE = True     # keep clips in Google Drive so a disconnect does not lose them
STEPS = 30               # quality vs speed (20 = faster, 40 = better)
FRAMES = 81              # 81 frames at 16 fps = about 5 seconds (must be 4*k+1)
GUIDANCE = 5.0
SEED = 42
ONLY = []                # e.g. [3, 5] to (re)make only clips 3 and 5; empty = all

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg

In [ ]:
import json, os
from google.colab import files

up = files.upload()  # pick clips.json
job = json.loads(next(iter(up.values())).decode("utf-8"))
print(job["title"], "|", job["ratio"], f'{job["width"]}x{job["height"]}', "|", len(job["clips"]), "clips")

OUT = "/content/clips"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    safe = "".join(c if c.isalnum() else "-" for c in job["title"])[:40] or "video"
    OUT = f"/content/drive/MyDrive/StickVoice/{safe}"
os.makedirs(OUT, exist_ok=True)
todo = [c for c in job["clips"] if (not ONLY or c["scene"] in ONLY)
        and (ONLY or not os.path.exists(f'{OUT}/{c["id"]}.mp4'))]
print("Saving to", OUT, "| clips to make:", [c["scene"] for c in todo])

In [ ]:
# Step 1: encode all prompts with the big text encoder, then free it (keeps the free GPU from running out of memory)
import gc, torch
from transformers import AutoTokenizer, UMT5EncoderModel

MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16  # T4 has no fast bf16

tok = AutoTokenizer.from_pretrained(MODEL, subfolder="tokenizer")
enc = UMT5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder", torch_dtype=DTYPE, device_map="cuda")

@torch.no_grad()
def embed(text):
    t = tok([text], padding="max_length", max_length=512, truncation=True, return_tensors="pt").to("cuda")
    e = enc(t.input_ids, t.attention_mask).last_hidden_state
    n = int(t.attention_mask.sum())
    e[:, n:] = 0  # Wan zeroes the padding, like WanPipeline.encode_prompt
    assert torch.isfinite(e).all(), "text encoder overflowed; try a shorter prompt"
    return e.to(DTYPE)

embeds = {c["id"]: (embed(c["prompt"]), embed(c["negative"])) for c in todo}
del enc; gc.collect(); torch.cuda.empty_cache()
print("Encoded", len(embeds), "prompts")

In [ ]:
# Step 2: load the video model (without the text encoder) and make the clips
from diffusers import AutoencoderKLWan, WanPipeline, UniPCMultistepScheduler
from diffusers.utils import export_to_video

vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=DTYPE)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)  # 3.0 suits 480p
pipe.to("cuda")
pipe.vae.enable_tiling()

import time
for c in todo:
    t0 = time.time()
    pos, neg = embeds[c["id"]]
    frames = pipe(prompt_embeds=pos, negative_prompt_embeds=neg, width=job["width"], height=job["height"],
                  num_frames=FRAMES, num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                  generator=torch.Generator("cuda").manual_seed(SEED + c["scene"])).frames[0]
    path = f'{OUT}/{c["id"]}.mp4'
    export_to_video(frames, path, fps=16)
    print(f'{c["id"]} done in {time.time() - t0:.0f}s ->', path)

In [ ]:
# Step 3: download every finished clip as clips.zip
import shutil
shutil.make_archive("/content/clips", "zip", OUT)
files.download("/content/clips.zip")

**Tips**
- A clip came out wrong? Set `ONLY = [n]` (and maybe change `SEED`), then run all again.
- "CUDA out of memory": lower `FRAMES` to 49 (3 s). Missing clips are filled with the StickVoice animation when assembling.
- Wan 2.1 makes silent video. The voice, sound effects and captions come from StickVoice's `audio.wav` and `captions.srt`.